# Homework 4 — Portfolio Q-learning desk (Module 4 framework)

**Salih Tutun, PhD** · DAT 5668 · Module 4

**Individual.** Due after **Session 8**.

Open this file in **Google Colab** from Canvas (or run it locally). When you are done, download the `.ipynb` and upload it on the Canvas assignment.

Session 8 practiced tabular Q-learning on an **inventory** desk. This homework keeps the **same Module 4 framework** on a **new finance desk**: a **SPY ETF portfolio** where each day you may **Buy**, **Hold**, or **Sell**. You DEFINE the MDP pieces, design a reward table that earns money when the market regime is read correctly, write `epsilon_greedy` + `q_update`, break the reward on purpose, and leave a framework line a CIO can read.

| Session skill | What you reuse here |
| --- | --- |
| 7 | Hand Q-update: $Q(s,a) \leftarrow Q(s,a) + \alpha(r + \gamma \max_{a'} Q(s',a') - Q(s,a))$ |
| 8 | `epsilon_greedy` (= decide) + `q_update` on consecutive market days |
| Framework | DEFINE → PIPELINE → interpret → human override |

**Data:** `portfolio_spy_daily.csv` — real SPY daily history (Yahoo Finance), **≥ 2000** trading days, **≥ 6** numeric features.

**Deliverable:** every `<<< FILL` completed and `review_hw4()` prints **PASS**.

---

## How to run this in Colab

1. Open from Canvas **Homework 4 — Open in Colab**, or use [this Colab file](https://colab.research.google.com/github/salihtutun/Agentic-AI_Autonomous_Decision_Making/blob/main/Students/Homework/HW4-Portfolio-Q-Learning.ipynb).
2. Run cells **top to bottom** (Shift+Enter). Do not skip the data cell.
3. Fill Parts 1–6. Re-run a cell after you edit it.
4. Run `review_hw4()` until it prints **PASS**.
5. File → Download → `.ipynb`. Upload on **Homework 4 — Portfolio Q-learning**.

If a cell says `NameError`, you skipped a cell above it. Run from the top.

## What “great” looks like

1. DEFINE names portfolio state / Buy-Hold-Sell / P&L language (not inventory shelves).
2. `epsilon_greedy` and `q_update` match the Module 4 update picture.
3. Your reward table makes **Bullish→Buy** and **Bearish→Sell** win after training.
4. The broken-reward experiment shows you can **read** a bad trading policy.
5. Framework line names the family (**RL**) and a **human override** (risk desk / CIO).


## Business story — SPY portfolio desk (finance)

You support a portfolio desk that trades the **SPY** ETF (U.S. large-cap equity exposure).

Each morning the desk observes a **market regime** from 10-day momentum, then chooses:

- **Buy** — add SPY exposure (deploy cash into the market)  
- **Hold** — keep the current book  
- **Sell** — reduce SPY exposure (raise cash / take profit / cut risk)  

| State (from `Momentum_10d`) | Meaning |
| --- | --- |
| Bearish Market | 10-day return $< -1\%$ |
| Neutral Market | between $-1\%$ and $+1\%$ |
| Bullish Market | 10-day return $> +1\%$ |

Dataset columns you will see (6+ features):

`Close`, `Daily_Return`, `Volatility_20d`, `Momentum_10d`, `Volume`, `RSI_14`, `Next_Day_Return`

```text
load SPY days
  → observe market regime s
  → decide a with ε-greedy on Q   (Buy / Hold / Sell)
  → reward r from your P&L-style table
  → next state s' = tomorrow's regime
  → q_update
  → repeat
```

A chatbot that never updates $Q$ is **not** doing Module 4. A policy that **Sells into a Bullish** regime is leaving money on the table — or worse, fighting the tape.


## Part 1 — DEFINE (before code)

Fill the next cell for the **portfolio** desk. Do not paste inventory / Clothing wording.

| Piece | Your job |
| --- | --- |
| State $s$ | What market signal do you observe before trading? |
| Actions $\mathcal{A}$ | List Buy, Hold, Sell |
| Reward $r$ | What money / risk score does the table represent? |
| Next state $s'$ | Where does tomorrow’s regime come from? |
| Not in $s$ | Name one document/exception a person still owns (e.g. earnings memo, compliance halt) |


In [ ]:
# ============================================================
# PART 1 — FILL (DEFINE in words)
# ============================================================

state_s = ""  # <<< FILL: market regime from momentum before the trade
actions = []  # <<< FILL: ["Buy", "Hold", "Sell"]
reward_r = ""  # <<< FILL: P&L / risk / return language
next_state = ""  # <<< FILL: next trading day's regime from the dataset
not_in_state = ""  # <<< FILL: e.g. compliance halt, CIO email, earnings memo

print("s:", state_s)
print("A:", actions)
print("r:", reward_r)
print("s':", next_state)
print("not in s:", not_in_state)


## Part 2 — LOAD (RUN WITH ME)

Loads **SPY** portfolio days and builds regimes from `Momentum_10d`. Do not edit the loader unless a path fails — then check Drive `MyDrive/DAT5668/data`.


In [ ]:
# ============================================================
# PART 2 — RUN WITH ME (SPY portfolio data)
# ============================================================

from pathlib import Path
from collections import Counter
import csv
import random

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
DATA_NAME = "portfolio_spy_daily.csv"
RNG = random.Random(7)


def course_data(name: str) -> Path:
    """Find course CSV on Drive, local assets, or GitHub raw fallback."""
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    here = Path.cwd()
    for base in (
        Path("data"),
        Path("assets/data"),
        Path("../assets/data"),
        Path("../../assets/data"),
        here / "assets" / "data",
        here.parent / "assets" / "data",
        Path("/content/data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    import urllib.request

    dest = Path("/content/data") if Path("/content").exists() else Path("data")
    dest.mkdir(parents=True, exist_ok=True)
    out = dest / name
    url = (
        "https://raw.githubusercontent.com/salihtutun/"
        "Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
    )
    urllib.request.urlretrieve(url, out)
    return out


def market_regime(momentum_10d: float) -> str:
    if momentum_10d < -0.01:
        return "Bearish Market"
    if momentum_10d <= 0.01:
        return "Neutral Market"
    return "Bullish Market"


ROWS = []
path = course_data(DATA_NAME)
with path.open(encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        mom = float(row["Momentum_10d"])
        ROWS.append(
            {
                "Date": row["Date"],
                "Close": float(row["Close"]),
                "Daily_Return": float(row["Daily_Return"]),
                "Volatility_20d": float(row["Volatility_20d"]),
                "Momentum_10d": mom,
                "Volume": int(float(row["Volume"])),
                "RSI_14": float(row["RSI_14"]),
                "Next_Day_Return": float(row["Next_Day_Return"]),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
                "State": market_regime(mom),
            }
        )

STATES = ["Bearish Market", "Neutral Market", "Bullish Market"]
ACTIONS = ["Buy", "Hold", "Sell"]
N_DAYS = len(ROWS)
FEATURE_COLS = [
    "Close",
    "Daily_Return",
    "Volatility_20d",
    "Momentum_10d",
    "Volume",
    "RSI_14",
    "Next_Day_Return",
]

print("File:", path.name)
print("Trading days:", N_DAYS, "(need ≥ 2000)")
print("Features (", len(FEATURE_COLS), "):", FEATURE_COLS)
print("Date range:", ROWS[0]["Date"], "→", ROWS[-1]["Date"])
print("Regime mix:", dict(Counter(r["State"] for r in ROWS)))
assert N_DAYS >= 2000, "Dataset too small — re-download portfolio_spy_daily.csv"
assert len(FEATURE_COLS) >= 6
print("Data ready. Next: design trading rewards, then write ε-greedy + Q-update.")


## Part 3 — Design the portfolio reward table (earn money when the regime is right)

In Session 8 the inventory rewards were given. Here **you** set the table so a portfolio desk would agree:

| Constraint (must hold) | Why |
| --- | --- |
| `R(Bullish, Buy) > R(Bullish, Sell)` | Do not fight a rising market |
| `R(Bearish, Sell) > R(Bearish, Buy)` | Cut risk / raise cash in a falling tape |
| Neutral: `Hold` should be best (or tied) | Avoid churn when the signal is flat |

Fill every entry. Then write `reward_rationale` (≥ 60 characters) explaining Bullish and Bearish choices in **money / risk** words.


In [ ]:
# ============================================================
# PART 3 — FILL reward table + rationale
# ============================================================

REWARDS = {
    ("Bearish Market", "Buy"): None,    # <<< FILL (should be worst or near-worst in Bearish)
    ("Bearish Market", "Hold"): None,   # <<< FILL
    ("Bearish Market", "Sell"): None,   # <<< FILL (should be best in Bearish)
    ("Neutral Market", "Buy"): None,    # <<< FILL
    ("Neutral Market", "Hold"): None,   # <<< FILL (best or tied in Neutral)
    ("Neutral Market", "Sell"): None,   # <<< FILL
    ("Bullish Market", "Buy"): None,    # <<< FILL (should be best in Bullish)
    ("Bullish Market", "Hold"): None,   # <<< FILL
    ("Bullish Market", "Sell"): None,   # <<< FILL (must be worse than Buy)
}

reward_rationale = ""  # <<< FILL ≥ 60 characters


def _row_ok():
    for k, v in REWARDS.items():
        if v is None:
            return False, f"missing reward for {k}"
    if not (REWARDS[("Bullish Market", "Buy")] > REWARDS[("Bullish Market", "Sell")]):
        return False, "Bullish: Buy must beat Sell"
    if not (REWARDS[("Bearish Market", "Sell")] > REWARDS[("Bearish Market", "Buy")]):
        return False, "Bearish: Sell must beat Buy"
    neu = [REWARDS[("Neutral Market", a)] for a in ACTIONS]
    if REWARDS[("Neutral Market", "Hold")] < max(neu):
        return False, "Neutral: Hold should be best (or tied)"
    if len(reward_rationale.strip()) < 60:
        return False, "reward_rationale needs ≥ 60 characters"
    return True, "reward table OK"


ok, msg = _row_ok()
print(msg)
for s in STATES:
    print(s, {a: REWARDS[(s, a)] for a in ACTIONS})


## Part 4 — PIPELINE: write `epsilon_greedy` + `q_update`

Same functions as Session 8 Stop 4 — now on consecutive **SPY** days.

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big)
$$


In [ ]:
# ============================================================
# PART 4 — FILL ε-greedy + Q-update, then train
# ============================================================

alpha = 0.3
epsilon = 0.5
gamma = 0.7
n_episodes = 40

Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}


def epsilon_greedy(state, Q, epsilon):
    """Decide: explore random action with prob epsilon, else greedy."""
    # <<< FILL
    return None


def q_update(Q, state, action, reward, next_state, alpha, gamma):
    """Update Q[state][action] in place with the TD target."""
    # <<< FILL
    return None


def train_q(rewards, seed=7):
    """Run the Module 4 pipeline on consecutive SPY regime days."""
    local_rng = random.Random(seed)
    Q_local = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
    for _ in range(n_episodes):
        for i in range(len(ROWS) - 1):
            state = ROWS[i]["State"]
            if local_rng.random() < epsilon:
                action = local_rng.choice(ACTIONS)
            else:
                action = max(Q_local[state], key=Q_local[state].get)
            reward = rewards[(state, action)]
            next_state = ROWS[i + 1]["State"]
            old = Q_local[state][action]
            Q_local[state][action] = old + alpha * (
                reward + gamma * max(Q_local[next_state].values()) - old
            )
    return Q_local


_hand_Q = {"Bullish Market": {"Buy": 0.0, "Hold": 0.0, "Sell": 0.0}}
try:
    a0 = epsilon_greedy("Bullish Market", _hand_Q, 0.0)
    q_update(_hand_Q, "Bullish Market", "Buy", 2.0, "Bullish Market", 0.5, 0.9)
    hand_ok = a0 == "Buy" and abs(_hand_Q["Bullish Market"]["Buy"] - 1.0) < 1e-9
except Exception as err:
    hand_ok = False
    print("NOT YET — fix epsilon_greedy / q_update:", err)

if hand_ok:
    Q = train_q(REWARDS)
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}
    agree = sum(1 for r in ROWS if policy[r["State"]] == r["Action_Taken"])
    print("Learned policy:", policy)
    print("Agree with heuristic Action_Taken:", agree, "/", N_DAYS, f"({agree / N_DAYS:.1%})")
    print(
        "CHECK 4 —",
        "PASS"
        if policy["Bullish Market"] == "Buy" and policy["Bearish Market"] == "Sell"
        else "LOOK — expected Bullish→Buy and Bearish→Sell with a sensible reward table",
    )
else:
    policy = None
    print("Fill epsilon_greedy and q_update, then re-run this cell.")


## Part 5 — Break the reward on purpose (interpret the trading policy)

Copy your table into `BROKEN`, then **flip** the Bullish row so **Sell** looks best (and Buy looks bad). Retrain. Compare policies.

Write `NOTE` (≥ 100 characters): what did the greedy portfolio policy do under the broken table, and why would a CIO refuse to deploy it?


In [ ]:
# ============================================================
# PART 5 — FILL broken rewards + NOTE
# ============================================================

BROKEN = dict(REWARDS)
# <<< FILL: make Bullish→Sell look best (example direction: Sell = +5, Buy = -1)
BROKEN[("Bullish Market", "Sell")] = None
BROKEN[("Bullish Market", "Buy")] = None

NOTE = ""  # <<< FILL ≥ 100 characters


def _broken_ready():
    if None in (BROKEN[("Bullish Market", "Sell")], BROKEN[("Bullish Market", "Buy")]):
        return False, "Set both BROKEN Bullish Sell and Bullish Buy numbers"
    if not (BROKEN[("Bullish Market", "Sell")] > BROKEN[("Bullish Market", "Buy")]):
        return False, "Broken table must prefer Sell over Buy when Bullish"
    if len(NOTE.strip()) < 100:
        return False, "NOTE needs ≥ 100 characters"
    return True, "broken setup OK"


bok, bmsg = _broken_ready()
print(bmsg)
if bok and hand_ok:
    Q_bad = train_q(BROKEN, seed=7)
    policy_bad = {s: max(Q_bad[s], key=Q_bad[s].get) for s in STATES}
    print("Policy with YOUR rewards:", policy)
    print("Policy with BROKEN rewards:", policy_bad)
    print(
        "CHECK 5 —",
        "PASS" if policy_bad["Bullish Market"] == "Sell" else "LOOK — broken Bullish should learn Sell",
    )
else:
    policy_bad = None


## Part 6 — Framework line (Module 4)

Name the family and one sentence you will keep. Also name a **human override** if a Q-row ever says **Sell on Bullish** (or Buy on Bearish) during a live week.


In [ ]:
# ============================================================
# PART 6 — FILL framework
# ============================================================

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL ≥ 40 characters
human_override = ""    # <<< FILL ≥ 40 characters — who stops a bad Sell-on-Bullish?

print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
print("Human override:", human_override or "(FILL)")


In [ ]:
# ============================================================
# REVIEW — run last. Must print PASS
# ============================================================

def review_hw4():
    """Student-facing checker for Homework 4 (portfolio)."""
    issues = []

    if len((state_s or "").strip()) < 12:
        issues.append("Part 1: state_s — describe the market regime / momentum state.")
    if set(a.lower() for a in (actions or [])) < {"buy", "hold", "sell"}:
        issues.append('Part 1: actions must include Buy, Hold, Sell.')
    if len((reward_r or "").strip()) < 10:
        issues.append("Part 1: reward_r — say what money/risk the table scores.")
    if len((next_state or "").strip()) < 10:
        issues.append("Part 1: next_state — tomorrow's market regime.")
    if len((not_in_state or "").strip()) < 8:
        issues.append("Part 1: not_in_state — name a document/exception path.")

    ok3, msg3 = _row_ok()
    if not ok3:
        issues.append("Part 3: " + msg3)

    try:
        probe = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
        probe["Bullish Market"]["Buy"] = 1.0
        greedy = epsilon_greedy("Bullish Market", probe, 0.0)
        if greedy != "Buy":
            issues.append("Part 4: epsilon_greedy(..., epsilon=0) must return the argmax action.")
        demoQ = {"Bullish Market": {"Buy": 0.0, "Hold": 0.0, "Sell": 0.0}}
        q_update(demoQ, "Bullish Market", "Buy", 2.0, "Bullish Market", 0.5, 0.0)
        if abs(demoQ["Bullish Market"]["Buy"] - 1.0) > 1e-9:
            issues.append("Part 4: q_update math — with γ=0, α=0.5, r=2 from 0 should become 1.0.")
    except Exception as err:
        issues.append(f"Part 4: epsilon_greedy/q_update error — {err}")

    if policy is None:
        issues.append("Part 4: train a policy (Bullish→Buy, Bearish→Sell) before review.")
    else:
        if policy.get("Bullish Market") != "Buy":
            issues.append("Part 4: with your REWARDS, Bullish Market should prefer Buy.")
        if policy.get("Bearish Market") != "Sell":
            issues.append("Part 4: with your REWARDS, Bearish Market should prefer Sell.")

    ok5, msg5 = _broken_ready()
    if not ok5:
        issues.append("Part 5: " + msg5)
    elif policy_bad is not None and policy_bad.get("Bullish Market") != "Sell":
        issues.append("Part 5: broken table should teach Bullish→Sell.")

    fam = (framework_family or "").strip()
    if fam not in {"LLM", "RL", "hybrid"}:
        issues.append("Part 6: framework_family must be LLM, RL, or hybrid.")
    if fam == "LLM":
        issues.append("Part 6: a numeric portfolio Q-table is RL (or hybrid with a human override) — not pure LLM.")
    if len((framework_line or "").strip()) < 40:
        issues.append("Part 6: framework_line needs ≥ 40 characters.")
    if len((human_override or "").strip()) < 40:
        issues.append("Part 6: human_override needs ≥ 40 characters.")

    if N_DAYS < 2000:
        issues.append("Data: need ≥ 2000 trading days in portfolio_spy_daily.csv.")
    if len(FEATURE_COLS) < 6:
        issues.append("Data: need ≥ 6 feature columns.")

    if issues:
        print("NOT YET")
        for issue in issues:
            print("-", issue)
        return False

    print("PASS")
    print("Policy:", policy)
    print(fam + ":", framework_line)
    return True


review_hw4()


## Submit checklist

1. [ ] Part 1 DEFINE filled for the **portfolio** desk (Buy / Hold / Sell)  
2. [ ] Part 3 reward inequalities hold + rationale ≥ 60 characters  
3. [ ] Part 4 `epsilon_greedy` + `q_update` → Bullish→Buy, Bearish→Sell  
4. [ ] Part 5 broken experiment + `NOTE` ≥ 100 characters  
5. [ ] Part 6 framework family + human override  
6. [ ] `review_hw4()` prints **PASS**

**Turn in:** this completed notebook on Canvas (**Homework 4 — Portfolio Q-learning**).
